# BSM L07G — Pochodzenie aplikacji, zaufanie w runtime i higiena kopii zapasowych (Android)

## Tryb pracy
To nie jest lab z Pythonem. Implementujesz rozwiązania w **Android Studio / Kotlin** w starterze projektu `lesson_g_app`.
Ten notebook służy jako:
- instrukcja krok-po-kroku (co otworzyć, gdzie kliknąć, czego szukać w kodzie),
- formularz odpowiedzi,
- mechanizm wysyłki odpowiedzi do backendu.

## Starter projektu
W tym repozytorium używasz folderu:
- `student/apps/lesson_g_app`

## Jak powstają odpowiedzi (ważne)
- **Zadanie 1 (G01)**: odpowiedź jest wysyłana automatycznie z aplikacji (nie ma w notebooku komórki „Wyślij”).
- **Zadania 2-4 (G02-G04)**: wypełniasz pole `#@param` i uruchamiasz komórkę, która wyśle odpowiedź.



In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


# G01 — Manifest i audyt prywatności (odpowiedź wysyła aplikacja)

## Cel
Zobaczyć w praktyce, że deklaracja w `AndroidManifest.xml` i rzeczywiste użycie funkcji to dwie różne rzeczy.
W prawdziwych review (np. aplikacje sklepowe, audyty prywatności) patrzy się na:
- listę uprawnień,
- powód biznesowy i techniczny,
- moment prośby o uprawnienie (runtime vs manifest),
- minimalizację zakresu (principle of least privilege).

## Co masz zrobić
1. Otwórz projekt `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik manifestu: `app/src/main/AndroidManifest.xml`.
1. Zrób mapę: dla każdego `<uses-permission ...>` zapisz, która funkcja aplikacji go realnie potrzebuje:
- lokalizacja: mapka (pobranie bieżącej lokalizacji)
- kamera: zrobienie zdjęcia
- galeria: wybór zdjęcia
- internet: pobranie kafelka mapy z usługi zewnętrznej
1. Odpowiedz sobie na pytania kontrolne (nie wysyłasz ich do backendu, ale są kluczowe do zrozumienia):
- Czy wszystkie zadeklarowane uprawnienia są faktycznie potrzebne? Jeśli tak, to w jakiej sytuacji?
- Które z nich są „runtime permissions” i kiedy aplikacja powinna o nie prosić?
- Czy aplikacja ma sensowny fallback, jeśli użytkownik odmówi?

## Jak zaliczasz (automatycznie)
To zadanie jest powiązane z aplikacją.
1. Uruchom aplikację na emulatorze lub urządzeniu.
1. Wpisz swoje **Student ID** w ekranie aplikacji (sekcja „Student / Task 1”).
1. Kliknij „Request permissions” i przejdź cały przepływ.
1. Jeśli wszystko jest poprawnie, aplikacja sama wykona wysyłkę dla `G01`.

Uwaga: w tym notebooku **nie ma** komórki „Wyślij” dla G01.


# G02 — APK / bundle provenance check: weryfikacja tożsamości builda

## Kontekst teoretyczny
Dwa APK mogą mieć ten sam `packageName`, ten sam UI, a nawet podobny kod, ale nadal nie są „tą samą aplikacją” z punktu widzenia zaufania.
Kluczowe rozróżnienie:
- **install-time trust**: system ufa aplikacji na podstawie podpisu podczas instalacji/aktualizacji.
- **runtime trust**: Twoja aplikacja (albo backend) podejmuje decyzję, czy w danym momencie ufać temu buildowi.

W tym zadaniu budujesz minimalny przepływ, który:
- odczytuje informację o podpisie (tożsamości) zainstalowanej aplikacji,
- porównuje ją z oczekiwaną tożsamością (wartością referencyjną w kodzie/testach),
- odrzuca build „nie ten co trzeba” (np. przepakowany/repackaged),
- potrafi wytłumaczyć różnicę: dlaczego sama nazwa paczki nie wystarcza.

## Co masz zrobić (krok po kroku)
1. Otwórz projekt `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
   Zwróć uwagę, że starter importuje `PackageManager` i `SigningInfo` (to jest trop pod provenance check).
1. Znajdź miejsce, w którym najlepiej umieścić weryfikację:
- przed wykonaniem operacji, której chcesz ufać (np. wysyłka / odblokowanie funkcji),
- i tak, żeby dało się to testować (warstwa helper, nie tylko UI).
1. Zaimplementuj prosty check podpisu:
- pobierz z `PackageManager` informacje o podpisie aplikacji,
- wyznacz stabilny identyfikator (np. skrót SHA-256 certyfikatu podpisującego),
- porównaj z wartością oczekiwaną w labie,
- jeśli nie pasuje: ustaw stan na „DENY” i przerwij akcję.

## Dokumentacja (konkretne API, których szukasz)
- `PackageManager`:
  https://developer.android.com/reference/android/content/pm/PackageManager
- `PackageManager.getPackageInfo(...)`:
  https://developer.android.com/reference/android/content/pm/PackageManager#getPackageInfo(java.lang.String,int)
- Flagi do podpisu (Android 9+): `PackageManager.GET_SIGNING_CERTIFICATES`
- `SigningInfo`:
  https://developer.android.com/reference/android/content/pm/SigningInfo
- `SigningInfo.getApkContentsSigners()`:
  https://developer.android.com/reference/android/content/pm/SigningInfo#getApkContentsSigners()
- `MessageDigest` (SHA-256):
  https://developer.android.com/reference/java/security/MessageDigest

## Jak zdobyć kod zaliczeniowy (G02)
1. Uruchom `./gradlew :app:bsmEvidence` w katalogu `student/apps/lesson_g_app`.
1. Jeśli G02 jest poprawne, w konsoli pojawi się 5-znakowy kod. Wklej go poniżej.



In [ ]:
#@title G02 — Wyślij odpowiedź
# Wklej 5-znakowy kod z `./gradlew :app:bsmEvidence`.
code_g02 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g02.strip())
print(final_answer)
zapisz_i_wyslij("G02", final_answer)



# G03 — Integrity-gated backend request: zaufanie w runtime + safe fallback

## Kontekst teoretyczny
Provenance check z G02 mówi „kim jest ten build”. To nadal nie wystarcza, jeśli backend ma przyjąć żądanie tylko od zaufanego klienta.
W tym zadaniu układasz przepływ:
- aplikacja wyznacza stan zaufania (verdict),
- żądanie jest powiązane z tą tożsamością (binding),
- przy braku zaufania aplikacja nie robi „best effort”, tylko bezpiecznie odmawia i tłumaczy użytkownikowi.

## Co masz zrobić (krok po kroku)
1. Otwórz: `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
   Zobacz strukturę `IntegrityState` i warunki na kod G03.
1. Otwórz: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
   Znajdź funkcję `submitAnswer(...)` i zobacz, co jest wysyłane.
1. Dodaj bramkę przed wysyłką:
- jeśli `verdict != "ALLOW"` albo binding nie jest spełniony, to nie wysyłaj,
- pokaż w UI czytelny status (np. w `banner` / `submissionStatus`).

## Dokumentacja (konkretne elementy z projektu)
- `HttpURLConnection`:
  https://developer.android.com/reference/java/net/HttpURLConnection
- `URL.openConnection()`:
  https://developer.android.com/reference/java/net/URL#openConnection()

## Jak zdobyć kod zaliczeniowy (G03)
1. Uruchom `./gradlew :app:bsmEvidence`.
1. Jeśli G03 jest poprawne, w konsoli pojawi się 5-znakowy kod. Wklej go poniżej.



In [ ]:
#@title G03 — Wyślij odpowiedź
# Wklej 5-znakowy kod z `./gradlew :app:bsmEvidence`.
code_g03 = ""  #@param {type:"string"}

final_answer = prepare_answer(code_g03.strip())
print(final_answer)
zapisz_i_wyslij("G03", final_answer)



# G04 — Higiena sekretów przy backupie i migracji (Auto Backup)

## Kontekst teoretyczny
Jeżeli aplikacja pozwala na backup, to system może przenieść część danych na inne urządzenie.
Dla sekretów to bywa krytyczne:
- rzeczy *device-only* nie powinny migrować,
- tokeny/sesje nie powinny odtwarzać się bez ponownego logowania,
- dane wrażliwe nie powinny trafiać do kopii.

## Co masz zrobić (krok po kroku)
1. Otwórz manifest: `app/src/main/AndroidManifest.xml`.
1. Oceń ustawienie `android:allowBackup` i zdecyduj:
- albo wyłączasz backup (`false`),
- albo zostawiasz backup i dodajesz reguły wykluczeń.
1. Jeśli robisz reguły, utwórz plik w `app/src/main/res/xml/` i podepnij go w `<application ...>`.
1. Zweryfikuj, że ścieżki/klucze używane na sekrety w labie nie migrują w sposób niezamierzony.

## Dokumentacja (konkretne atrybuty manifestu)
- `android:allowBackup`:
  https://developer.android.com/guide/topics/manifest/application-element#allowbackup
- `android:fullBackupContent`:
  https://developer.android.com/guide/topics/manifest/application-element#fullBackupContent
- `android:dataExtractionRules`:
  https://developer.android.com/guide/topics/manifest/application-element#dataExtractionRules

## Skąd wziąć odpowiedź
W starterze jest 5-znakowa wartość sekretu dla G04.
1. Otwórz `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Wyszukaj `TASK_4_SECRET_...`.
1. Odczytaj wartość w sposób przewidziany przez lab i wklej ją poniżej.



In [ ]:
#@title G04 — Wyślij odpowiedź
# Wklej 5-znakową wartość sekretu dla zadania 4.
secret_g04 = ""  #@param {type:"string"}

final_answer = prepare_answer(secret_g04.strip())
print(final_answer)
zapisz_i_wyslij("G04", final_answer)

